#### Extraction des caractéristiques avec model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt

Ce script réalise l’extraction de caractéristiques compactes (4096 dimensions) à partir d’images satellites à l’aide d’un modèle Fully Convolutional basé sur VGG16. Voici les étapes principales expliquées :

Chargement des données et préparation des chemins : Le script commence par définir les chemins des fichiers nécessaires, notamment le modèle sauvegardé (model_path), le répertoire contenant les images de test (test_image_dir), le fichier CSV contenant les métadonnées associées aux images (csv_path), et l'emplacement où sauvegarder le fichier final avec les caractéristiques extraites (output_path).

Définition du modèle Fully Convolutional : Une classe VGGFullyConv est définie pour adapter l'architecture VGG16. Les couches convolutives (self.features) extraient des caractéristiques riches, et les couches de classification convolutive (self.classifier) compressent ces caractéristiques à 4096 dimensions via deux convolutions 1x1. Une réduction spatiale moyenne (mean(dim=(2, 3))) est ensuite appliquée pour condenser les informations spatiales.

Chargement du modèle et mise en mode évaluation : Le modèle est instancié et ses poids sont chargés depuis un fichier pré-entraîné. Il est ensuite transféré sur un GPU ou CPU selon la disponibilité, et configuré pour le mode évaluation (désactivation du calcul des gradients).

Transformation des images : Un pipeline de transformation est défini pour normaliser les images en entrée afin qu'elles soient compatibles avec VGG16. Les images sont redimensionnées à 224x224 pixels, converties en tenseurs, et normalisées en utilisant les moyennes et écarts-types des canaux RGB du jeu ImageNet.

Vérification des dimensions des caractéristiques : Une image factice (aléatoire) est passée dans le modèle pour vérifier que la sortie après réduction est de taille [1, 4096]. Cette étape permet de s'assurer que le modèle fonctionne comme prévu.

Extraction des caractéristiques compactes : Pour chaque image du fichier CSV, le script :

Vérifie si le fichier image existe dans le répertoire.
Charge et transforme l’image avec le pipeline défini.
Passe l’image transformée dans le modèle Fully Convolutional pour extraire un vecteur de caractéristiques compactes de taille 4096.
Si une image est manquante ou corrompue, un vecteur de zéros est utilisé par défaut.
Fusion des caractéristiques avec le fichier CSV : Les vecteurs de caractéristiques extraits pour chaque image sont combinés avec les métadonnées existantes du fichier CSV. Chaque vecteur (4096 dimensions) est ajouté comme nouvelles colonnes, nommées feature_0 à feature_4095.

Sauvegarde des résultats : Le DataFrame final, contenant les métadonnées et les caractéristiques extraites, est sauvegardé dans un fichier CSV à l’emplacement spécifié (output_path). Le script confirme la sauvegarde par un message.

En résumé, ce script transforme des images satellites en vecteurs de caractéristiques compactes de 4096 dimensions, facilitant ainsi leur utilisation pour des tâches d’analyse ou de modélisation ultérieures. Les étapes de gestion des erreurs et de fusion garantissent la robustesse et l’intégrité des résultats.

In [1]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models

# Chemins des fichiers et répertoires
model_path = r"D:\wealth_predict_2021\models\model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt"
test_image_dir = r"D:\wealth_predict_2021\data\downloaded\Image_satellite_EHCVM_2021_Zoom_18_Image_2024"
csv_path = r"D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images.csv"
output_path = r"D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv"

# Définir l'architecture du modèle Fully Convolutional
class VGGFullyConv(nn.Module):
    def __init__(self, num_classes=4):
        super(VGGFullyConv, self).__init__()
        self.features = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, num_classes, kernel_size=1)  # Nombre de classes
        )

    def forward(self, x):
        x = self.features(x)  # Extraction des caractéristiques convolutives
        x = self.classifier[:2](x)  # Appliquer les convolutions 1x1 jusqu'à la production de 4096 caractéristiques
        return x.mean(dim=(2, 3))  # Réduction spatiale (moyenne sur hauteur et largeur)

# Charger le DataFrame contenant les métadonnées
df = pd.read_csv(csv_path)

# Charger le modèle sauvegardé
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VGGFullyConv(num_classes=4).to(device)  # Charger l'architecture
model.load_state_dict(torch.load(model_path), strict=False)  # Charger les poids
model.eval()  # Mettre le modèle en mode évaluation

# Transformation des images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Fonction pour charger une image
def load_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        return transform(image)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

# Vérification de la taille des caractéristiques
dummy_image = torch.randn(1, 3, 224, 224).to(device)
dummy_features = model(dummy_image)  # Passer une image factice dans le modèle
print(f"Dimensions des caractéristiques compactes : {dummy_features.shape}")  # Résultat attendu : [1, 4096]

# Extraction des caractéristiques compactes
def extract_image_features(model, image_tensor):
    image_tensor = image_tensor.unsqueeze(0).to(device)  # Ajouter une dimension pour le batch
    with torch.no_grad():
        features = model(image_tensor)  # Passer par le modèle Fully Convolutional
    return features.cpu().numpy().flatten()  # Retourner les caractéristiques compactes comme tableau numpy

# Liste pour stocker les caractéristiques des images
features_list = []
for idx, row in df.iterrows():
    image_name = row['nom de l\'image']
    image_path = os.path.join(test_image_dir, image_name)

    if os.path.exists(image_path):  # Vérifier si l'image existe
        image_tensor = load_image(image_path)
        if image_tensor is not None:
            features = extract_image_features(model, image_tensor)
        else:
            features = np.zeros(4096)  # Vecteur par défaut si erreur
    else:
        print(f"Image introuvable : {image_name}")
        features = np.zeros(4096)  # Vecteur par défaut si l'image n'existe pas

    features_list.append(features)

# Conversion des caractéristiques en DataFrame
features_df = pd.DataFrame(features_list, columns=[f"feature_{i}" for i in range(4096)])

# Combiner les caractéristiques avec le DataFrame original
df_with_features = pd.concat([df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)

# Sauvegarder le DataFrame final
df_with_features.to_csv(output_path, index=False)
print(f"DataFrame avec caractéristiques compactes (4096) sauvegardé sous : {output_path}")


Dimensions des caractéristiques compactes : torch.Size([1, 4096])
DataFrame avec caractéristiques compactes (4096) sauvegardé sous : D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv


In [ ]:
pd.read_csv(r"D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv")

In [ ]:
pd.read_csv(r"D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv").columns[1:56,]